# TELE 6510 protocol performance

This notebook collects the results of the repository's CoAP, HTTP, and MQTT tests and stores one run per execution. It uses the existing shell scripts. It does not reimplement the 30-transaction trials.

Two modes, set in the next code cell:

- `MODE = "ingest"` reads a results tree the scripts already wrote (`lab_results/` by default).
- `MODE = "live"` runs `environment-1-coap-http/test.sh` and `environment-2-coap-mqtt/test.sh`, then ingests `lab_results/`.

Live mode needs `coap-client`, `coap-server`, `curl`, `http-server`, Mosquitto, `tshark`, `bc`, `tc`, and passwordless `sudo` for `tc netem` on `lo`. If any of that is missing, the notebook prints why and does not start the scripts. It then ingests `lab_results/` when that directory has CSVs or captures, otherwise the labeled example fixture.

`notebooks/sample_lab_results/` is **example data**. The latencies are synthetic, there are 3 transactions instead of 30, and there are no captures. Rows from that tree are marked `example_data`. Do not put them in the lab report.

## Install

`tshark` is a system package, already installed by `utils/install-dependencies.sh`. Python packages:

```bash
python3 -m pip install -r requirements.txt
jupyter notebook notebooks/protocol_performance.ipynb
```

Run Jupyter from the repository root or from `notebooks/`.

The same notebook runs inside the Compose stack. Bring that up with `docker compose up -d --build --wait` and open <http://127.0.0.1:8888>. The sweep cell below stays off unless you set `RUN_DOCKER_SWEEP`.


## What the lab asks for, and what the scripts record

Environment 1 is device to edge to application with CoAP and HTTP. Environment 2 is the same path with CoAP and MQTT.

| Lab question | Where it comes from |
| --- | --- |
| Environment 1. Average request/response delay for a readout of 10 bytes or less, 30 transactions, Wireshark timestamps | Confirmable CoAP GET `/time`, then HTTP GET `/time`. The headline is the end-to-end mean at 0% loss. The HTTP-only mean is the application protocol's own request/response. |
| Environment 1. Packet loss versus latency for confirmable and non-confirmable CoAP at 0%, 3%, 5%, 10% | The CoAP-only trials, not the combined trial. |
| Environment 2. Average device-to-application delay, 30 transactions, Wireshark timestamps | Confirmable CoAP GET, then MQTT publish. Reported separately for QoS 0, 1, and 2. The 0% rows are the headline. |
| Environment 2. Packet loss versus latency for confirmable CoAP and MQTT QoS 0, 1, and 2 | CoAP-only confirmable trial and the MQTT-only trials. QoS 0 / 1 / 2 are at most once, at least once, and exactly once. |
| Environment 2. CoAP-based MQTT versus HTTP under the same loss | Combined confirmable CoAP + MQTT versus the HTTP-only trial, plus the CoAP + HTTP end-to-end path. |

The scripts also capture with `tshark -i lo -w <file>.pcapng`. This notebook parses those files with a field export:

```bash
tshark -r capture.pcapng \
  -d udp.port==5683,coap -d tcp.port==8080,http -d tcp.port==1883,mqtt \
  -Y "coap or http or mqtt" -T fields \
  -e frame.time_epoch -e coap.type -e coap.mid -e coap.token \
  -e http.request.method -e http.response.code \
  -e mqtt.msgtype -e mqtt.qos -e mqtt.msgid -e mqtt.msg \
  -E separator=, -E quote=d -E header=y
```

Timing rules, applied only when the frames are actually in the capture:

- CoAP: request to the response with the same token, else the same message id, else the next response. Latency is the difference of `frame.time_epoch`.
- HTTP: request to the response on the same `tcp.stream`. Port 8080 is decoded as HTTP because that is the port `test.sh` uses.
- MQTT device-to-application: client `PUBLISH` (destination port 1883) to the broker `PUBLISH` forwarded to the subscriber (source port 1883). The QoS handshake (`PUBACK` or `PUBCOMP`) is stored beside that and is not substituted for it.
- Combined end-to-end: CoAP request epoch to the HTTP response or MQTT delivery epoch, paired in order.

`test.sh` also logs a client clock (`date +%s.%N` around each command). That includes process startup. It is stored as `script_latency_ms`. The lab asks for Wireshark timestamps, so the tshark columns are the ones to report when they are present. The script columns remain so a run is still usable when tshark or the capture is missing. Empty cells are missing data, not zero latency.

Packet loss is the configured `tc netem` rate on `lo`, not a count of dropped application messages. The scripts do not record throughput. CoAP and HTTP body sizes are not fixed by the scripts; `coap_payload_bytes` and `http_payload_bytes` are filled only from tshark (`data.len`, `http.content_length`). MQTT payloads configured in `test.sh` are `date +%Y-%m-%d` (10 bytes) for the MQTT-only trial and the word `timestamp` (9 bytes) for the combined trial. Those configured sizes are labeled separately from measured sizes.

The raw `.pcapng` is copied or hardlinked into the run directory. Open it in Wireshark for the trace screenshots the lab asks you to paste. The files are gitignored.

### Messages the scripts exchange

Environment 1, each transaction: `coap-client -m get coap://[::1]:5683/time` (add `-N` for non-confirmable), and `curl http://localhost:8080/time`. The combined trial does CoAP and then HTTP. Environment 2: `mosquitto_sub -h localhost -t iot/time -q <qos>`, a confirmable CoAP GET, and `mosquitto_pub -h localhost -t iot/time -q <qos>`. The MQTT-only publish body is the date. The combined publish body is `timestamp`. QoS 0, 1, and 2 are the three MQTT qualities in the lab question. Describing *your* capture is still a Wireshark step; the preview cell below only prints a short text listing when tshark can read a file.


## Configure

Leave the defaults to ingest. Point `SOURCE` at a `lab_results` directory if it is not the checkout you started Jupyter from. `STORE` is the versionable run directory (`notebooks/run-store` by default). Set `FORCE = True` to store another copy of an unchanged source. Set `INCLUDE_EXAMPLE = False` once you have a real run and want the plots to hide the synthetic fixture.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent]
for candidate in candidates:
    if (candidate / "iot_results.py").is_file():
        candidate_text = str(candidate)
        if candidate_text not in sys.path:
            sys.path.insert(0, candidate_text)
        break
else:
    raise RuntimeError("Open this notebook from the repository root or from notebooks/.")

import iot_results as iot

MODE = "ingest"  # "ingest" or "live"
SOURCE = None
STORE = None
FORCE = False
INCLUDE_EXAMPLE = True

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda value: "" if pd.isna(value) else f"{value:.3f}")
%matplotlib inline


## Docker sweep

Leave `RUN_DOCKER_SWEEP = False` for ingest or for the host `test.sh` live mode. Set it to `True` only when `docker compose up -d --build --wait` has already succeeded and this kernel can see the Docker socket (the Jupyter service mounts it). The cell runs `sh docker/sweep.sh`. It does not run `docker compose up`.

`LOSS_PCTS`, `TRANSACTIONS`, `SLEEP_S`, and `SWEEP` (`env1`, `env2`, or `both`) are the same variables as the shell script. The default is the lab grid: 0/3/5/10%, 30 transactions, both environments. Loss is `tc netem` on the device egress and the edge access egress. Captures land in `lab_results/*/captures/` as `.pcapng`. The following collect cell then prefers that tree over the example fixture.

The access-side capture sees CoAP, HTTP, and the MQTT handshake. It does not see the broker PUBLISH forwarded to the subscriber, so the tshark MQTT delivery column can be empty. The CSV client clock is still stored. Details and the Desktop limits are in the repository README.


In [ ]:
import os
import subprocess

RUN_DOCKER_SWEEP = False
SWEEP = "both"  # env1, env2, or both
LOSS_PCTS = "0 3 5 10"
TRANSACTIONS = "30"
SLEEP_S = "1"

if not RUN_DOCKER_SWEEP:
    print("RUN_DOCKER_SWEEP is False. No containers were started and no loss was applied.")
else:
    sweep = iot.repo_root() / "docker" / "sweep.sh"
    env = os.environ.copy()
    env["SWEEP"] = SWEEP
    env["LOSS_PCTS"] = str(LOSS_PCTS)
    env["TRANSACTIONS"] = str(TRANSACTIONS)
    env["SLEEP_S"] = str(SLEEP_S)
    print("Running", sweep)
    completed = subprocess.run(["sh", str(sweep)], cwd=iot.repo_root(), env=env, check=False)
    if completed.returncode != 0:
        raise RuntimeError(f"docker/sweep.sh exited {completed.returncode}")
    print("Sweep finished. The next cell ingests lab_results/ when MODE is ingest.")


## Tool check

This does not start brokers or change packet loss. `tshark -D` is only a permission hint for live capture. Reading an existing `.pcapng` does not need capture permission.


In [ ]:
tools = iot.check_tools()
tool_table = pd.DataFrame(
    [{"tool": name, "path": path or ""} for name, path in tools["tools"].items()]
)
display(tool_table)
print("passwordless sudo for tc/netem:", tools["sudo_noninteractive"])
print("tshark:", tools["tshark"]["version"] or tools["tshark"]["reason"])
print("capture:", tools["capture_note"])
print("live environments that can start:", tools["live_ready_environments"] or "(none)")
if tools["live_blockers"]:
    print("Live mode will not start until these are resolved:")
    for blocker in tools["live_blockers"]:
        print("-", blocker)


## Collect one run

A run stores `metadata.json`, `transactions.csv`, `summary.csv`, and `lab_tables.json`. Captures are linked under `captures/` and named from `metadata.json`. Re-running on the same files reuses the run unless `FORCE` is set.


In [ ]:
result = iot.collect(
    mode=MODE,
    source=SOURCE,
    store=STORE,
    force=FORCE,
)
store = Path(result["run_dir"]).parent
print("run_id:", result["run_id"])
print("run_dir:", result["run_dir"])
print("example_data:", result["example_data"])
print("reused:", result["reused"])
for warning in result["warnings"]:
    print("WARNING:", warning)

meta = iot.load_metadata(store, result["run_id"])
print("git:", meta.get("git"))
print("host:", meta.get("host"))
print("tshark command:", meta["tshark"]["field_command"])
print("captures:", len(meta.get("captures") or []))


## Past runs

Each row is one stored execution. `example_data` is true for the synthetic fixture. `capture_count` is the number of `.pcapng` files kept with that run.


In [ ]:
index = iot.load_index(store)
display(index)

transactions = iot.load_transactions(store)
summary = iot.summarize_transactions(transactions)
if not INCLUDE_EXAMPLE:
    summary = summary.loc[~summary["example_data"].map(iot.as_bool)].copy()
if summary.empty:
    print("No runs to show. Set INCLUDE_EXAMPLE = True or ingest a lab_results directory.")

def _mean_columns(frame):
    return [column for column in frame.columns if column.endswith("_ms")]

def show_lab_table(title, frame):
    print("\n" + title)
    means = _mean_columns(frame)
    if frame.empty or not means or not any(frame[column].notna().any() for column in means):
        print("No values for this timing source. Wireshark columns stay empty until a capture is parsed with tshark.")
    display(frame)


## Environment 1

Question 1 is the average delay. The device-to-application headline is confirmable CoAP followed by HTTP, at 0% packet loss. The HTTP row at 0% is the application request/response by itself. Question 2 is packet loss versus latency for confirmable and non-confirmable CoAP. The tshark table is the Wireshark-timestamp answer. The script table is the client clock from `test.sh`.


In [ ]:
env1_q1_script = iot.env1_q1_delay(summary, "script")
env1_q1_tshark = iot.env1_q1_delay(summary, "tshark")
env1_q2_script = iot.env1_q2_loss_table(summary, "script")
env1_q2_tshark = iot.env1_q2_loss_table(summary, "tshark")

print(iot.headline_sentence(env1_q1_script, "Environment 1"))
print(iot.headline_sentence(env1_q1_tshark, "Environment 1"))
show_lab_table("Environment 1 question 1, script clock", env1_q1_script)
show_lab_table("Environment 1 question 1, Wireshark timestamps (tshark)", env1_q1_tshark)
show_lab_table("Environment 1 question 2, packet loss vs CoAP, script clock", env1_q2_script)
show_lab_table("Environment 1 question 2, packet loss vs CoAP, Wireshark timestamps (tshark)", env1_q2_tshark)


## Environment 2, and MQTT versus HTTP

Question 1 is the device-to-application delay for each MQTT QoS. The 0% rows are the average the lab asks for. Question 2 is packet loss versus confirmable CoAP and MQTT QoS 0, 1, and 2. Question 4 compares CoAP-based MQTT (the combined confirmable CoAP + MQTT trial) with HTTP at the same configured loss. `coap_http_e2e_mean_ms` is the Environment 1 combined path, included so both full paths can be read together. Each table uses one timing source.


In [ ]:
env2_q1_script = iot.env2_q1_delay(summary, "script")
env2_q1_tshark = iot.env2_q1_delay(summary, "tshark")
env2_q2_script = iot.env2_q2_loss_table(summary, "script")
env2_q2_tshark = iot.env2_q2_loss_table(summary, "tshark")
q4_script = iot.env2_q4_mqtt_vs_http(summary, "script")
q4_tshark = iot.env2_q4_mqtt_vs_http(summary, "tshark")

print(iot.headline_sentence(env2_q1_script, "Environment 2"))
print(iot.headline_sentence(env2_q1_tshark, "Environment 2"))
show_lab_table("Environment 2 question 1, script clock", env2_q1_script)
show_lab_table("Environment 2 question 1, Wireshark timestamps (tshark)", env2_q1_tshark)
show_lab_table("Environment 2 question 2, packet loss vs CoAP and MQTT QoS, script clock", env2_q2_script)
show_lab_table("Environment 2 question 2, packet loss vs CoAP and MQTT QoS, Wireshark timestamps (tshark)", env2_q2_tshark)
show_lab_table("Question 4, CoAP-based MQTT vs HTTP, script clock", q4_script)
show_lab_table("Question 4, CoAP-based MQTT vs HTTP, Wireshark timestamps (tshark)", q4_tshark)


## Plots

Lines are mean latency against the configured packet loss. A run built from the example fixture is labeled `example`. The Wireshark figure is drawn only when tshark produced numbers. Figures are also written under the current run's `figures/` directory.


In [ ]:
def _plot_lines(frame, columns, title):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plotted = False
    if not frame.empty:
        for column, label in columns:
            if column not in frame.columns:
                continue
            for run_id, part in frame.groupby("run_id", sort=False):
                if not part[column].notna().any():
                    continue
                example = bool(part["example_data"].iloc[0]) if "example_data" in part.columns else False
                prefix = "example " if example else ""
                ax.plot(
                    part["packet_loss_pct"],
                    part[column],
                    marker="o",
                    label=f"{prefix}{run_id} {label}",
                )
                plotted = True
    ax.set_xlabel("Packet loss (%)")
    ax.set_ylabel("Mean latency (ms)")
    ax.set_title(title)
    ax.set_xticks([0, 3, 5, 10])
    ax.grid(True, alpha=0.3)
    if plotted:
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "No samples for this timing source", ha="center", va="center", transform=ax.transAxes)
    fig.tight_layout()
    return fig

def _save(fig, name):
    figure_dir = Path(result["run_dir"]) / "figures"
    figure_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(figure_dir / name, dpi=120)
    plt.show()

_save(
    _plot_lines(
        env1_q2_script,
        [
            ("coap_confirmable_mean_ms", "CoAP confirmable"),
            ("coap_nonconfirmable_mean_ms", "CoAP non-confirmable"),
        ],
        "Environment 1: CoAP latency vs packet loss (script clock)",
    ),
    "env1_q2_script.png",
)
if env1_q2_tshark["coap_confirmable_mean_ms"].notna().any() or env1_q2_tshark["coap_nonconfirmable_mean_ms"].notna().any():
    _save(
        _plot_lines(
            env1_q2_tshark,
            [
                ("coap_confirmable_mean_ms", "CoAP confirmable"),
                ("coap_nonconfirmable_mean_ms", "CoAP non-confirmable"),
            ],
            "Environment 1: CoAP latency vs packet loss (Wireshark timestamps)",
        ),
        "env1_q2_tshark.png",
    )
else:
    print("Skipped the Environment 1 Wireshark plot: tshark latencies are empty.")

_save(
    _plot_lines(
        env2_q2_script,
        [
            ("coap_confirmable_mean_ms", "CoAP confirmable"),
            ("mqtt_qos0_mean_ms", "MQTT QoS 0"),
            ("mqtt_qos1_mean_ms", "MQTT QoS 1"),
            ("mqtt_qos2_mean_ms", "MQTT QoS 2"),
        ],
        "Environment 2: CoAP and MQTT latency vs packet loss (script clock)",
    ),
    "env2_q2_script.png",
)
if any(env2_q2_tshark[column].notna().any() for column in (
    "coap_confirmable_mean_ms", "mqtt_qos0_mean_ms", "mqtt_qos1_mean_ms", "mqtt_qos2_mean_ms"
)):
    _save(
        _plot_lines(
            env2_q2_tshark,
            [
                ("coap_confirmable_mean_ms", "CoAP confirmable"),
                ("mqtt_qos0_mean_ms", "MQTT QoS 0"),
                ("mqtt_qos1_mean_ms", "MQTT QoS 1"),
                ("mqtt_qos2_mean_ms", "MQTT QoS 2"),
            ],
            "Environment 2: CoAP and MQTT latency vs packet loss (Wireshark timestamps)",
        ),
        "env2_q2_tshark.png",
    )
else:
    print("Skipped the Environment 2 Wireshark plot: tshark latencies are empty.")

fig, ax = plt.subplots(figsize=(8, 4.5))
q4_plot_cols = [
    ("http_mean_ms", "HTTP"),
    ("coap_http_e2e_mean_ms", "CoAP+HTTP"),
    ("coap_mqtt_qos0_e2e_mean_ms", "CoAP+MQTT QoS 0"),
    ("coap_mqtt_qos1_e2e_mean_ms", "CoAP+MQTT QoS 1"),
    ("coap_mqtt_qos2_e2e_mean_ms", "CoAP+MQTT QoS 2"),
]
if not q4_script.empty and any(q4_script[column].notna().any() for column, _label in q4_plot_cols):
    latest = q4_script["run_id"].iloc[-1]
    part = q4_script[q4_script["run_id"] == latest]
    example = bool(part["example_data"].iloc[0])
    x = range(len(q4_plot_cols))
    # One grouped series per loss rate, for the latest run only.
    for loss in (0, 3, 5, 10):
        row = part[part["packet_loss_pct"] == loss]
        if row.empty:
            continue
        ax.plot(
            list(x),
            [row[column].iloc[0] for column, _label in q4_plot_cols],
            marker="o",
            label=f"{loss}% loss",
        )
    ax.set_xticks(list(x))
    ax.set_xticklabels([label for _column, label in q4_plot_cols], rotation=15, ha="right")
    title = "CoAP-based MQTT vs HTTP (script clock)"
    if example:
        title = "EXAMPLE DATA — " + title
    ax.set_title(title + f"\n{latest}")
    ax.set_ylabel("Mean latency (ms)")
    ax.grid(True, alpha=0.3)
    ax.legend(title="Configured loss")
else:
    ax.text(0.5, 0.5, "No question 4 samples", ha="center", va="center", transform=ax.transAxes)
fig.tight_layout()
_save(fig, "q4_mqtt_vs_http_script.png")


## Wireshark traces

The lab asks for traces pasted from Wireshark. The `.pcapng` files listed here open directly in Wireshark; that is the file to screenshot. A text preview is printed only when `tshark` can read the capture. It is not a substitute for the screenshot, and it is empty for the example fixture because that fixture has no captures.


In [ ]:
captures = meta.get("captures") or []
if not captures:
    print("This run has no .pcapng files.")
    reason = meta.get("tshark", {}).get("reason") or ""
    if reason:
        print(reason)
    print("After a real test.sh run, ingest lab_results/. Captures are stored under the run's captures/ directory.")
else:
    for capture in captures:
        path = Path(result["run_dir"]) / capture["stored_as"]
        print(f"{path}  {capture['bytes']} bytes  link={capture['link_method']}")
        if capture.get("tshark_error"):
            print("  tshark:", capture["tshark_error"])
    preview_path = Path(result["run_dir"]) / captures[0]["stored_as"]
    print()
    print(iot.tshark_text_preview(preview_path, limit=20))


## Files in this run

Commit the CSV and JSON if you want the run in git. Leave the `.pcapng` files and `figures/` untracked (they match `.gitignore`). Column definitions are in `notebooks/run-store/README.md`.

To collect a real lab session after the shell tests:

1. Run `./comprehensive-test/lab_test.sh` or either environment `test.sh` on a machine with the brokers and `tshark`.
2. Set `SOURCE` to that `lab_results` directory, or leave it as `None` if `lab_results/` is in this checkout.
3. Run the notebook again with `INCLUDE_EXAMPLE = False`.
4. Open `captures/*.pcapng` in Wireshark for the report screenshots.
5. Use the tshark tables when they are filled. Use the script-clock tables only when the Wireshark columns are empty, and say that in the report. Check `n` against the lab's 30 transactions and check the payload columns against the 10-byte readout.


In [ ]:
run_dir = Path(result["run_dir"])
for path in sorted(run_dir.rglob("*")):
    if path.is_file() and path.suffix not in {".pcap", ".pcapng", ".png"}:
        print(f"{path.relative_to(run_dir)}  {path.stat().st_size} bytes")
